# 00. 파이썬 기본 라이브러리 둘러보기

스터디에서 쓰는 라이브러리 8개를 **우리 데이터(logP)** 로 하나씩 써 본다. 셀마다 Shift + Enter.

| 라이브러리 | 하는 일 | 이 스터디에서 |
|---|---|---|
| `numpy` | 숫자 배열 계산 | 특징 행렬, 오차 계산 |
| `pandas` | 표(DataFrame) 다루기 | CSV 읽기, 정리, 결과표 |
| `matplotlib` | 그림 | 분포, parity plot, 학습곡선 |
| `rdkit` | 분자 다루기 | SMILES → 분자, 기술자, 지문, 그림 |
| `scikit-learn` | 고전 머신러닝 | 분할, Ridge · RF · SVR, 평가 지표 |
| `xgboost` | 부스팅 트리 | 1주차 비교 모델 |
| `torch` | 신경망 | GNN 의 계산 엔진 |
| `torch_geometric` | 그래프 신경망 | 분자 그래프, GIN |

설치가 안 되어 있으면 `00_설치안내.md` 부터.

## 1. numpy: 배열

In [ ]:
import numpy as np
a = np.array([1.0, 2.0, 3.0, 4.0])
print('평균', a.mean(), ' 표준편차', a.std().round(3), ' 제곱', a ** 2)

M = np.random.default_rng(0).normal(size=(3, 4))   # 3행 4열 = 분자 3개 x 특징 4개
print('모양', M.shape)
print('열 평균', M.mean(axis=0).round(2))           # axis=0 : 행 방향으로(분자들에 대해) 평균

> 핵심: 반복문 없이 배열 전체에 한 번에 계산한다. `axis=0` 은 "행을 따라", `axis=1` 은 "열을 따라".

## 2. pandas: 표

In [ ]:
import pandas as pd
df = pd.read_csv('data/logP.csv')
print(df.shape)
df.head()

In [ ]:
print(df['value'].describe().round(2))       # 열 하나 요약
print(df['source'].value_counts())           # 범주 세기

In [ ]:
small = df[df['value'] > 5]                  # 조건으로 행 고르기
print(f'logP > 5 인 분자: {len(small):,}개')
small.sort_values('value', ascending=False).head(3)

## 3. matplotlib: 그림

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(df['value'], bins=50, color='#0072B2')
ax.set_xlabel('logP'); ax.set_ylabel('count')
plt.tight_layout(); plt.show()

> 그림은 항상 `fig, ax = plt.subplots()` 로 시작해서 `ax.` 에 그린다. 축 이름은 반드시 단다.

## 4. RDKit: 분자

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator

mol = Chem.MolFromSmiles('CCO')                    # 에탄올
print('원자 수', mol.GetNumAtoms(), ' 분자량', round(Descriptors.MolWt(mol), 2))
print('canonical SMILES:', Chem.MolToSmiles(Chem.MolFromSmiles('OCC')))   # 같은 분자는 같은 글자로

mfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp = mfp.GetFingerprintAsNumPy(mol)
print('Morgan 지문 길이', len(fp), ' 켜진 비트', fp.sum())

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df['smiles'].head(6)]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(200, 160),
                     legends=[f'logP {v:.2f}' for v in df['value'].head(6)])

## 5. scikit-learn: 고전 머신러닝 (분할 → 학습 → 평가)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

sub = df.sample(1000, random_state=0)
X = np.array([[Descriptors.MolWt(m), Descriptors.TPSA(m), Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m)]
              for m in sub['smiles'].apply(Chem.MolFromSmiles)])
y = sub['value'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

model = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_tr, y_tr)   # 모든 모델이 fit
pred = model.predict(X_te)                                                         # 모든 모델이 predict
print(f'기술자 4개 + RF : test MAE = {mean_absolute_error(y_te, pred):.3f}')
print(f'평균으로 찍기   : test MAE = {mean_absolute_error(y_te, np.full_like(y_te, y_tr.mean())):.3f}')

> scikit-learn 의 모든 모델은 `fit(X, y)` 로 배우고 `predict(X)` 로 예측한다. 1주차 노트북의 모델 6개도 같다.

## 6. XGBoost: 같은 사용법, 다른 모델

In [ ]:
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=0).fit(X_tr, y_tr)
print(f'XGBoost        : test MAE = {mean_absolute_error(y_te, xgb.predict(X_te)):.3f}')

## 7. PyTorch: 텐서와 자동 미분

In [ ]:
import torch
w = torch.tensor(2.0, requires_grad=True)     # 학습할 값
x, y_true = torch.tensor(3.0), torch.tensor(9.0)
loss = (w * x - y_true) ** 2                   # 손실 = (예측 - 정답)^2
loss.backward()                                # 미분 자동 계산
print('손실', loss.item(), ' dL/dw', w.grad.item())   # 2 (wx - y) x = 2 (6 - 9) 3 = -18

> 신경망 학습 = 이 미분(기울기) 방향으로 `w` 를 조금씩 고치는 것(경사하강법). GNN 도 똑같다, 단지 `w` 가 수십만 개다.

## 8. PyTorch Geometric: 분자 그래프

In [ ]:
from feature import smiles_to_graph
from torch_geometric.data import Data
x, edge_index, edge_attr = smiles_to_graph('CCO')
g = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
print(g)                                       # 원자 3개, 엣지 4개 (결합 2개 x 양방향)
print('edge_index =', edge_index.tolist())

> 1주차 과제의 `python train.py --prop 팀물성` 은 이 그래프를 수천 개 만들어 GIN 에 넣고 학습한다.
> 다음 순서: `01_multi_model_comparison.ipynb`